# Cassava Leaf Disease Detection — Model & Training (Colab)

Custom CNN trained **entirely from scratch** (no pretrained weights / transfer
learning, per project requirements).

**Classes (5):** CBB, CBSD, CGM, Healthy, CMD

This notebook combines `model.py` (architecture) and `train.py` (Member 1's
real data pipeline, with the split/transform bugs fixed + class-weighted loss
for the imbalance found in EDA) into one runnable Colab notebook.


## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
import os

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- in Colab, go to Runtime > Change runtime type > GPU")


## 2. Get the dataset

Run whichever cell matches how Member 1's notebook gets the data (Kaggle
download, Google Drive mount, etc.) and set `DATA_DIR` to point at the
resulting folder. Expected structure:

```
cassava_dataset/
├── Cassava___bacterial_blight/
├── Cassava___brown_streak_disease/
├── Cassava___green_mottle/
├── Cassava___healthy/
└── Cassava___mosaic_disease/
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

# This path is specific to Member 1's Google Drive -- if the zip isn't at this
# exact location in YOUR Drive, update it (or ask Member 1 to share the folder
# so the path matches, or copy the zip into your own Drive at the same path).
zip_path = "/content/drive/MyDrive/Cassava_Project/data.zip"
extract_path = "/content/cassava_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))


## 3. Model architecture (`model.py`)

5 conv blocks (conv → ReLU → maxpool), each halving spatial size:
`224 → 112 → 56 → 28 → 14 → 7`, then flatten → FC → dropout → FC (raw
logits — `nn.CrossEntropyLoss` applies log-softmax internally).

No BatchNorm (not covered in the course's CNN lab) — overfitting is managed
with dropout + Member 1's augmentation instead, since there's no pretrained
model to lean on.

Layers are named directly (`conv1`, `conv2`, ...) rather than hidden in a
`Sequential` block, so filters/feature maps can be inspected the same way
as in the course's CNN lab (e.g. `model.conv1.weight`, forward hooks).


In [ ]:
# Order matches torchvision's ImageFolder alphabetical class_to_idx assignment
# from Member 1's notebook:
#   {'Cassava___bacterial_blight': 0, 'Cassava___brown_streak_disease': 1,
#    'Cassava___green_mottle': 2, 'Cassava___healthy': 3, 'Cassava___mosaic_disease': 4}
# Keeping this order is what makes index 2 mean "CGM" consistently across the
# model, the evaluation/confusion-matrix code, and the API.
CLASS_NAMES = ["CBB", "CBSD", "CGM", "Healthy", "CMD"]
NUM_CLASSES = len(CLASS_NAMES)


class CassavaCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(256 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 224 -> 112
        x = self.pool(F.relu(self.conv2(x)))   # 112 -> 56
        x = self.pool(F.relu(self.conv3(x)))   # 56  -> 28
        x = self.pool(F.relu(self.conv4(x)))   # 28  -> 14
        x = self.pool(F.relu(self.conv5(x)))   # 14  -> 7

        x = x.view(x.size(0), -1)              # flatten: [batch, 256*7*7]
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)                        # raw logits
        return x


def build_model(num_classes=NUM_CLASSES):
    return CassavaCNN(num_classes=num_classes)


In [ ]:
# Self-check: architecture summary + parameter count (same by-hand exercise as Lab 3)
model = build_model()
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

fc1_by_hand = (256 * 7 * 7) * 256 + 256
print(f"fc1 by-hand check: {fc1_by_hand:,} "
      f"(matches fc1: {sum(p.numel() for p in model.fc1.parameters()) == fc1_by_hand})")


## 4. Data pipeline (`train.py`, fixed)

Fixes two bugs from the original notebook:

1. `train_dataset` was being redefined over the **full** dataset, silently
   discarding the 80/10/10 split (data leakage).
2. `val_dataset`/`test_dataset` pointed at an untransformed base dataset —
   no `ToTensor()`, so `DataLoader` would crash trying to batch raw PIL images.

**Fix:** split first (indices only), then wrap each split in `TransformSubset`
so train gets augmentation and val/test get a clean resize+tensor only.


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32  # matches Member 1's DataLoader batch size

# Same augmentation as Member 1's notebook -- train only.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
])

# No augmentation for val/test -- evaluate on real, undistorted images.
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


In [ ]:
class TransformSubset(Dataset):
    """
    Wraps a torch Subset (a split of indices into the base ImageFolder) and
    applies a chosen transform on __getitem__ -- lets each split (train/val/test)
    get its own transform after splitting ONCE, instead of reloading the whole
    dataset per split like the original notebook did.
    """

    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]  # base ImageFolder returns a PIL image here
        if self.transform is not None:
            image = self.transform(image)
        return image, label


In [ ]:
def build_dataloaders(data_dir, batch_size=BATCH_SIZE):
    # No transform here -- split first, apply transforms per-split below.
    base_dataset = datasets.ImageFolder(root=data_dir)
    print("Total images:", len(base_dataset))
    print("class_to_idx:", base_dataset.class_to_idx)

    # Sanity check: make sure CLASS_NAMES above still lines up with whatever
    # ImageFolder assigns here (alphabetical by folder name).
    expected_order = [name.replace("Cassava___", "") for name in sorted(base_dataset.classes)]
    print("CLASS_NAMES:      ", CLASS_NAMES)
    print("ImageFolder order:", expected_order)

    train_size = int(0.80 * len(base_dataset))
    val_size = int(0.10 * len(base_dataset))
    test_size = len(base_dataset) - train_size - val_size

    generator = torch.Generator().manual_seed(42)  # same seed as the original notebook
    train_split, val_split, test_split = random_split(
        base_dataset, [train_size, val_size, test_size], generator=generator
    )
    print(f"Train: {len(train_split)}  Val: {len(val_split)}  Test: {len(test_split)}")

    train_dataset = TransformSubset(train_split, train_transform)
    val_dataset = TransformSubset(val_split, eval_transform)
    test_dataset = TransformSubset(test_split, eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Class weights for the loss, computed from the TRAIN split only (avoids
    # leaking val/test distribution into training).
    class_counts = torch.zeros(NUM_CLASSES)
    for idx in train_split.indices:
        _, label = base_dataset.samples[idx]
        class_counts[label] += 1
    print("Train class counts:", class_counts.tolist())

    class_weights = 1.0 / class_counts.clamp(min=1)
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # normalize

    return train_loader, val_loader, test_loader, class_weights


## 5. Training utilities

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


## 6. Configuration

(In the `.py` version these came from `argparse` command-line flags -- in a
notebook, just set them directly as variables here.)


In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3
CHECKPOINT_DIR = "/content/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_DIR}. "
        f"Run the dataset-download cell in section 2 first, or update DATA_DIR."
    )


## 7. Build data loaders + model

In [ ]:
train_loader, val_loader, test_loader, class_weights = build_dataloaders(DATA_DIR)

model = build_model().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


## 8. Train

Saves `best_model.pt` to `CHECKPOINT_DIR` whenever validation accuracy improves.


In [ ]:
best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch}/{EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
        torch.save(model.state_dict(), ckpt_path)
        print(f"  -> New best val acc ({va_acc:.3f}), saved to {ckpt_path}")


## 9. Final test evaluation

Run this **once**, only after training/tuning is completely finished --
repeated test-set checks while tuning would let you unintentionally overfit
to it, defeating the purpose of a held-out test set.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Final test loss {test_loss:.3f} | test acc {test_acc:.3f}")
